In [1]:
import numpy as np
import scipy.optimize 

#### Numerical Optimization and L-BFGS

Numerical optimization plays a fundamental role in training classification models, such as Logistic Regression. Unlike Gaussian models, for which closed-form expressions are available for ML parameter estimation, logistic regression requires iterative minimization of the objective function.

#### Gradient Descent

Consider the minimization of a function $f(\theta)$. The gradient descent method iteratively updates the parameter vector:

$$\theta^{(k+1)} = \theta^{(k)} - \alpha \nabla f(\theta^{(k)})$$

where $\alpha > 0$ is the learning rate (step size).

**Limitations**: convergence depends heavily on the choice of $\alpha$; too small leads to slow convergence, too large may cause divergence.

#### L-BFGS

**L-BFGS** (Limited-memory Broyden–Fletcher–Goldfarb–Shanno) is a quasi-Newton optimization method that approximates the inverse Hessian using a limited history of gradient evaluations. It combines the fast convergence of second-order methods with the low memory footprint of first-order methods.

Key advantages:
- Automatically adapts the step size using curvature information
- Does not require explicit computation or storage of the Hessian matrix
- Converges much faster than gradient descent in practice

We use `scipy.optimize.fmin_l_bfgs_b`, which also supports **box constraints** (lower/upper bounds on individual parameters).

#### Gradient Computation

Given the objective function $f(\theta)$, the gradient $\nabla f(\theta)$ is the vector of partial derivatives:

$$\nabla f(\theta) = \left[\frac{\partial f}{\partial \theta_1}, \frac{\partial f}{\partial \theta_2}, \ldots, \frac{\partial f}{\partial \theta_n}\right]$$

If the gradient is not provided to the optimizer, `scipy` approximates it numerically via finite differences:

$$\frac{\partial f}{\partial \theta_i} \approx \frac{f(\theta + h \, e_i) - f(\theta)}{h}$$

where $h$ is a small perturbation and $e_i$ is the $i$-th standard basis vector. This is convenient but slower and less accurate than an analytical gradient.

##### Practical Example

We demonstrate the use of `scipy.optimize.fmin_l_bfgs_b` on a simple quadratic function $f(x, y) = (x - y)^2 + x^2$ to verify correct usage before applying it to logistic regression.

In [2]:
def function(x):
    y=x[0]
    z=x[1]
    return (y+3)**2 + np.sin(y) + (z+1)**2
def grad(x):
    y=x[0]
    z=x[1]

    dy=2*(y+3)+np.cos(y)
    dz=2*(z+1)
    return np.array([dy,dz])

In [3]:
x0 = np.array([0.0,0.0])
x, f, d = scipy.optimize.fmin_l_bfgs_b(func=function, x0=x0, fprime=grad)

print("Punto di minimo (x):", x)
print("Valore della funzione (f):", f)
print("Dizionario delle informazioni (d):", d)

Punto di minimo (x): [-2.57747137 -0.99999927]
Valore della funzione (f): -0.3561430123647611
Dizionario delle informazioni (d): {'grad': array([-1.50318729e-06,  1.46120529e-06]), 'task': 'CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL', 'funcalls': 7, 'nit': 6, 'warnflag': 0}


#### Binary Logistic Regression

Binary logistic regression is a probabilistic classification model used to predict membership in one of two mutually exclusive classes. Unlike linear regression, which produces continuous values, logistic regression transforms the linear predictor through the **sigmoid function**:

$$\sigma(s) = \frac{1}{1 + e^{-s}}, \quad s = w^T x + b$$

The model outputs a posterior probability $P(C=1 \mid x) = \sigma(w^T x + b)$.

**Training objective**: we minimize the regularized cross-entropy (negative log-likelihood):

$$J(w, b) = \frac{\lambda}{2} \|w\|^2 + \frac{1}{N} \sum_{i=1}^{N} \left[ z_i \log \sigma(s_i) + (1 - z_i) \log(1 - \sigma(s_i)) \right]$$

where $\lambda$ is the L2 regularization strength and $z_i \in \{0, 1\}$ are the binary labels.

A **prior-weighted** variant adjusts the loss contributions to target a specific class prior $\pi_T$, which is useful when the operating conditions differ from the training class balance.

> See implementation in [`scripts/logisticreg.py`](../scripts/logisticreg.py): functions `trainLogReg` (standard) and `weighted_trainLogReg` (prior-weighted).
> Evaluation metrics: [`scripts/bayes_error.py`](../scripts/bayes_error.py) — functions `DCF_norm`, `compute_min_dcf`, `get_bayes_decision`.

In [20]:
%load_ext autoreload
%autoreload 2
from scripts.utils import *
from scripts.logisticreg import *
from scripts.bayes_error import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
D, L = load_iris_binary()
(DTR, LTR), (DVAL, LVAL) = split_db_2to1(D, L)

In [21]:
lambdas=[1e-3,1e-1,1.0]
pi_emp = np.sum(LTR == 1) / LTR.shape[0]
emp_prior_log_odds = np.log(pi_emp / (1 - pi_emp))

for l in lambdas:
    v_opt,J_min=trainLogReg(DTR,LTR,l)
    w_opt=v_opt[:-1]
    b_opt=v_opt[-1]
    
    S_val=np.dot(w_opt.T,DVAL)+b_opt
    LP=(S_val>0).astype(int)
    error_rate = np.mean(LP != LVAL)
    llr=S_val-emp_prior_log_odds
    min_dcf=compute_min_dcf(llr,LVAL,0.5,1,1)

    predictions_bayes = get_bayes_decision(llr, 0.5, 1, 1)
    act_dcf = DCF_norm(predictions_bayes, LVAL, 0.5, 1, 1)

    print(f"Lambda: {l}")
    print(f"J ottima: {J_min:e}")
    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")

Lambda: 0.001
J ottima: 1.100009e-01
Error Rate: 8.8%
DCF min: 0.0625
Actual DCF:0.11805555555555555

Lambda: 0.1
J ottima: 4.539407e-01
Error Rate: 11.8%
DCF min: 0.05555555555555555
Actual DCF:0.1111111111111111

Lambda: 1.0
J ottima: 6.316436e-01
Error Rate: 14.7%
DCF min: 0.1111111111111111
Actual DCF:0.16666666666666666



In [22]:
lambdas=[1e-3,1e-1,1.0]
pi_T = 0.8
target_prior_log_odds = np.log(pi_T / (1 - pi_T))

for l in lambdas:
    v_opt,J_min=weighted_trainLogReg(DTR,LTR,l,pi_T)
    w_opt=v_opt[:-1]
    b_opt=v_opt[-1]
    
    S_val=np.dot(w_opt.T,DVAL)+b_opt
    LP=(S_val>0).astype(int)
    error_rate = np.mean(LP != LVAL)
    llr=S_val-target_prior_log_odds
    min_dcf=compute_min_dcf(llr,LVAL,pi_T,1,1)

    predictions_bayes = get_bayes_decision(llr, pi_T, 1, 1)
    act_dcf = DCF_norm(predictions_bayes, LVAL, pi_T, 1, 1)

    print(f"Lambda: {l}")
    print(f"J ottima: {J_min:e}")
    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")

Lambda: 0.001
J ottima: 9.401035e-02
Error Rate: 11.8%
DCF min: 0.16666666666666666
Actual DCF:0.2222222222222222

Lambda: 0.1
J ottima: 3.606261e-01
Error Rate: 38.2%
DCF min: 0.05555555555555555
Actual DCF:0.7222222222222222

Lambda: 1.0
J ottima: 4.724715e-01
Error Rate: 52.9%
DCF min: 0.1111111111111111
Actual DCF:1.0

